# Análise exploratória — PaySim (TRANSFER e CASH_OUT)

Objetivo: conhecer os dados preparados por `python -m src.prepare_data` antes do treino, respondendo:

1. Quão desbalanceadas são as classes?
2. O valor (`amount`) e a hora do dia (`hour`) diferem entre fraudes e legítimas?
3. Como as transações se distribuem ao longo dos dias? (importa para a divisão temporal)

A análise é só descritiva: nenhum modelo é ajustado aqui. Os gráficos são salvos em
`reports/figures/` para o relatório e a apresentação.

Lembrete: a base é **simulada** e **não brasileira**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# O notebook roda de dentro de notebooks/; a raiz do projeto é a pasta acima
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.config import PROCESSED_FILE, REPORTS_DIR, TARGET, TEMPORAL_SPLIT_STEP

FIGURAS = REPORTS_DIR / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED_FILE)
print(df.shape)
df.head()

## 1. Desbalanceamento das classes

In [ ]:
contagem = pd.crosstab(df["type"], df[TARGET], margins=True, margins_name="total")
contagem.columns = ["legítima", "fraude", "total"]
contagem["% fraude"] = (100 * contagem["fraude"] / contagem["total"]).round(3)
contagem

In [ ]:
# Um "modelo" que sempre responde "legítima" já teria esta acurácia:
print(f"Acurácia de quem nunca aponta fraude: {1 - df[TARGET].mean():.2%}")

Por isso a acurácia **não** é a métrica principal: ela fica alta mesmo sem detectar nenhuma
fraude. As fraudes se dividem quase igualmente entre TRANSFER e CASH_OUT, mas, como há muito mais
CASH_OUT legítimos, a proporção de fraude é bem maior em TRANSFER.

## 2. Valor da transação (`amount`)

In [ ]:
resumo_valor = df.groupby(TARGET)["amount"].describe()
resumo_valor.index = ["legítima", "fraude"]
resumo_valor.round(2)

In [ ]:
# Os valores vão de centavos a dezenas de milhões, então o gráfico usa log10 do valor.
# Valores iguais a zero ficam de fora (não existe log de zero).
print("Transações com amount = 0 (fora do gráfico):", (df["amount"] == 0).sum())

com_valor = df[df["amount"] > 0]
log_valor = np.log10(com_valor["amount"])
bins = np.linspace(log_valor.min(), log_valor.max(), 60)

fig, ax = plt.subplots(figsize=(8, 4))
for classe, nome, cor in [(0, "legítima", "tab:blue"), (1, "fraude", "tab:red")]:
    ax.hist(log_valor[com_valor[TARGET] == classe], bins=bins, density=True,
            alpha=0.5, label=nome, color=cor)
ax.set_xlabel("log10(amount)   (5 = 100 mil, 6 = 1 milhão, 7 = 10 milhões)")
ax.set_ylabel("densidade")
ax.set_title("Distribuição do valor por classe")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURAS / "valor_por_classe.png", dpi=150)
plt.show()

In [ ]:
fraudes = df[df[TARGET] == 1]
legitimas = df[df[TARGET] == 0]
print("Maior valor de fraude:", fraudes["amount"].max())
print("Fraudes com valor exatamente 10 milhões:", (fraudes["amount"] == 10_000_000).sum())
print("Legítimas acima de 10 milhões:", (legitimas["amount"] > 10_000_000).sum())

- As fraudes tendem a ter valores mais altos (compare as medianas na tabela), mas as duas
  distribuições se sobrepõem bastante: o valor sozinho não separa as classes.
- Nenhuma fraude passa de 10 milhões, e várias têm exatamente esse valor, enquanto há legítimas
  acima disso. Parece um limite do simulador; a base não explica. Fica registrado como possível
  artefato da simulação.

## 3. Hora do dia (`hour`)

`hour = step % 24`, calculada em `src/prepare_data.py`.

In [ ]:
por_hora = pd.crosstab(df["hour"], df[TARGET])
por_hora.columns = ["legítima", "fraude"]
por_hora["% fraude"] = (100 * por_hora["fraude"] / (por_hora["legítima"] + por_hora["fraude"])).round(2)
por_hora

In [ ]:
# Esquerda: em que horas cada classe acontece (cada linha soma 100%).
# Direita: entre as transações de cada hora, quantas são fraude.
distribuicao = pd.crosstab(df["hour"], df[TARGET], normalize="columns") * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(distribuicao.index, distribuicao[0], marker="o", label="legítima", color="tab:blue")
ax1.plot(distribuicao.index, distribuicao[1], marker="o", label="fraude", color="tab:red")
ax1.set_xlabel("hora do dia")
ax1.set_ylabel("% das transações da classe")
ax1.set_title("Quando cada classe acontece")
ax1.legend()

ax2.bar(por_hora.index, por_hora["% fraude"], color="tab:red")
ax2.set_xlabel("hora do dia")
ax2.set_ylabel("% de fraude na hora")
ax2.set_title("Proporção de fraude por hora")

fig.tight_layout()
fig.savefig(FIGURAS / "hora_por_classe.png", dpi=150)
plt.show()

- As fraudes acontecem em quantidade parecida em todas as horas; as legítimas se concentram
  durante o dia. Por isso, de madrugada, a proporção de fraude fica muito alta.
- A hora tende a ser uma variável forte, mas esse padrão pode ser em parte efeito do simulador.
- **Ponto para o treino:** a relação não é "quanto maior a hora, maior o risco"; o risco sobe de
  madrugada e cai de dia. O Random Forest captura isso sozinho. A Regressão Logística, recebendo a
  hora como um número de 0 a 23, só consegue representar uma tendência crescente ou decrescente,
  o que tende a prejudicá-la nessa comparação.

## 4. Transações ao longo dos dias

In [ ]:
# step é a hora simulada (1 a 744): dia 1 = steps 1 a 24, dia 2 = steps 25 a 48, ...
dia = (df["step"] - 1) // 24 + 1
por_dia = pd.crosstab(dia, df[TARGET])
por_dia.index.name = "dia"
por_dia.columns = ["legítima", "fraude"]
por_dia

In [ ]:
# Linha tracejada: corte da divisão temporal (treino até o step TEMPORAL_SPLIT_STEP)
corte_dia = TEMPORAL_SPLIT_STEP / 24 + 0.5

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.bar(por_dia.index, por_dia["legítima"], color="tab:blue")
ax1.set_ylabel("legítimas")
ax1.set_title("Transações por dia")
ax2.bar(por_dia.index, por_dia["fraude"], color="tab:red")
ax2.set_ylabel("fraudes")
ax2.set_xlabel("dia da simulação")
for ax in (ax1, ax2):
    ax.axvline(corte_dia, color="black", linestyle="--")
ax1.text(corte_dia + 0.3, ax1.get_ylim()[1] * 0.85, "início do teste temporal")

fig.tight_layout()
fig.savefig(FIGURAS / "transacoes_por_dia.png", dpi=150)
plt.show()

- As fraudes são praticamente constantes todos os dias.
- As legítimas despencam a partir do dia 18; os dias 3 a 5 também têm poucas. O dia 31
  (steps 721 a 743, onde a base termina) não tem nenhuma legítima, só fraudes.
- É uma característica do simulador; a base não explica o motivo.

## 5. Consequência para a divisão temporal

In [ ]:
no_treino = df["step"] <= TEMPORAL_SPLIT_STEP
divisao = pd.crosstab(no_treino.map({True: "treino", False: "teste"}), df[TARGET])
divisao.columns = ["legítima", "fraude"]
divisao["% fraude"] = (100 * divisao["fraude"] / (divisao["legítima"] + divisao["fraude"])).round(2)
divisao

**Decisão do grupo:** manter o corte (treino nos dias 1 a 23, teste nos dias 24 a 31) e
registrar a diferença como limitação.

O que isso significa na leitura dos resultados:

- O teste temporal tem uma proporção de fraude cerca de 11 vezes maior que o treino.
- Precision e PR-AUC dependem dessa proporção: com mais fraude no teste, a Precision tende a
  subir mesmo que o modelo não melhore. Por isso os modelos devem ser comparados **dentro** de cada
  divisão, e não a divisão temporal contra a estratificada.
- Para dar uma referência, reportar junto da PR-AUC a proporção de fraude do teste (é a PR-AUC de
  um modelo que chuta ao acaso).

## Conclusões

1. Classes muito desbalanceadas: a acurácia engana, então as métricas principais são PR-AUC,
   Precision, Recall e F1.
2. Fraudes têm valores mais altos em geral, mas com muita sobreposição; há um teto de 10 milhões
   nas fraudes, provavelmente do simulador.
3. A hora do dia mostra o padrão mais marcante: fraudes constantes, legítimas concentradas de dia.
   Esse padrão não é linear, o que favorece o Random Forest frente à Regressão Logística.
4. A quantidade de legítimas cai muito a partir do dia 18; o corte temporal foi mantido e isso
   entra como limitação.

Todas essas observações valem para uma base **simulada** e **não brasileira**.